<a href="https://colab.research.google.com/github/ShiX27/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


In [3]:
# Count number of rows in the dataset
raw_rows = len(df)
print('starting with', raw_rows, 'rows')
print()

# take inventory
print(df.info())
print()
print(df.describe())
print()
print('shape:', df.shape)
print()
print('dtypes:', df.dtypes)
print()
print('nulls:', df.isnull().sum())
print()
print('duplicates:', df.duplicated().sum())

starting with 315 rows

<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 0 to 122
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  315 non-null    int64  
 1   item      315 non-null    object 
 2   category  315 non-null    object 
 3   qty       302 non-null    float64
 4   price     315 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 14.8+ KB
None

         order_id         qty
count  315.000000  302.000000
mean   150.638095    1.486755
std     86.242118    0.913987
min      0.000000   -1.000000
25%     76.500000    1.000000
50%    150.000000    1.000000
75%    224.500000    2.000000
max    299.000000    3.000000

shape: (315, 5)

dtypes: order_id      int64
item         object
category     object
qty         float64
price        object
dtype: object

nulls: order_id     0
item         0
category     0
qty         13
price        0
dtype: int64

duplicates: 15


**Code Explanation:** This step examines the original `df` DataFrame before any cleaning so I can better understand the dataset and decide what needs to be cleaned. First, compute and print the number of rows in the original `df` DataFrame and store it in `raw_rows`. Second, use `.info()` to view the columns, non-null counts, and data types. Third, use `.describe()` to summarize the numeric columns (`order_id` and `qty`), including count, mean, standard deviation, minimum, 25th percentile, median, 75th percentile, and maximum. Fourth, print the number of rows and columns using `.shape`. Fifth, print the data type of each column using `.dtypes`. Sixth, print the number of missing values (nulls) for each column. Finally, print the number of duplicate rows.

### TODO 1 — drop duplicates

In [4]:
removed = df.duplicated().sum()   # count how many duplicates
clean = df.drop_duplicates().copy()   # drop duplicate rows

print('Remaining', len(clean), 'rows')

# create log(...)
log('drop duplicates', 'dropped exact duplicate rows', removed)

Remaining 300 rows
[drop duplicates] dropped exact duplicate rows (15 row(s))


**Code Explanation:** First, use `df.duplicated().sum()` to count the number of duplicate rows and store the result in the `removed` variable. Next, drop the duplicate rows from the `df` DataFrame and make a copy named `clean`. Finally, print the number of remaining rows and create a log entry to record the cleaning step and the number of rows affected.

### TODO 2 — clean `price` -> float

In [5]:
clean['price'] = (clean['price'].astype(str)
                  .str.replace('$', '', regex=False)    # remove $ signs
                  .str.replace(',', '', regex=False)    # remove commas
                  .str.strip()
                  .astype(float))

print(clean['price'].dtype)

print('Remaining', len(clean), 'rows')

# create log(...)
log('clean price', 'price arrived as text; converted to float', len(clean))

float64
Remaining 300 rows
[clean price] price arrived as text; converted to float (300 row(s))


**Code Explanation:** First, convert the `price` column in the `clean` DataFrame to float by temporarily converting the values to strings, removing dollar signs and commas, stripping any leading or trailing spaces, and then converting the cleaned values to float. Next, print the data type of the `price` column to confirm the conversion. Finally, print the number of remaining rows, which should not change during this step, and create a log entry to record the cleaning step and the number of rows affected.

In [6]:
# Revenue before and after dropping duplicates

before_price = (
    df['price'].astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .astype(float)
)

before_duplicates = (before_price * df['qty']).sum()
after_duplicates = (clean['price'] * clean['qty']).sum()
change_in_revenue = before_duplicates - after_duplicates

print(f'Before: ${before_duplicates:.2f}')
print(f'After: ${after_duplicates:.2f}')
print(f'Change in revenue: ${change_in_revenue:.2f}')

Before: $4852.50
After: $4594.50
Change in revenue: $258.00


The revenue before dropping duplicates is $4852.50.

The revenue after dropping duplicates is $4594.50.

The change in revenue is $258.00.

**Code Explanation:** This step calculates the total revenue before and after dropping duplicate rows and the resulting change in revenue. First, convert the `price` column in the original `df` DataFrame to float by temporarily converting the values to strings, removing dollar signs and commas, stripping any leading or trailing spaces, and then converting the cleaned values to float. Next, calculate the total revenue before and after dropping duplicates and store the results in `before_duplicates` and `after_duplicates`, respectively. Then, calculate the change in revenue by subtracting `after_duplicates` from `before_duplicates`. Finally, print the revenue before and after dropping duplicates, along with the change in revenue, rounded to two decimal places.

### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [7]:
# convert the "qty" values into numbers
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # count NaN quantities
negative = (clean['qty'] < 0).sum()   # count negative quantities

# TODO 3a: drop missing qty
before_missing = (clean['price'] * clean['qty']).sum()
clean = clean[clean['qty'].notna()].copy()
after_missing = (clean['price'] * clean['qty']).sum()
change_missing = before_missing - after_missing

print(f'Before dropping missing qty: ${before_missing:.2f}')
print(f'After dropping missing qty: ${after_missing:.2f}')
print(f'Change in revenue: ${change_missing:.2f}')

# TODO 3b: drop negative qty
before_negative = after_missing
clean = clean[clean['qty'] >= 0].copy()
after_negative = (clean['price'] * clean['qty']).sum()
change_negative = after_negative - before_negative

print(f'Before dropping negative qty: ${before_negative:.2f}')
print(f'After dropping negative qty: ${after_negative:.2f}')
print(f'Change in revenue: ${change_negative:.2f}')

print('Remaining', len(clean), 'rows')

# log both separately
log('drop missing qty', 'dropped rows with no quantity', missing)
log('drop negative qty', 'dropped rows with negative quantity', negative)

Before dropping missing qty: $4594.50
After dropping missing qty: $4594.50
Change in revenue: $0.00
Before dropping negative qty: $4594.50
After dropping negative qty: $4740.00
Change in revenue: $145.50
Remaining 275 rows
[drop missing qty] dropped rows with no quantity (12 row(s))
[drop negative qty] dropped rows with negative quantity (13 row(s))


The revenue before and after dropping rows with missing qty is the same, $4594.50.

So, the change in revenue is $0.00.

Dropping rows with missing quantity did not change the calculable total revenue because revenue could not be determined for those rows without a quantity value.

The revenue before dropping rows with negative qty is $4594.50.

The revenue after dropping rows with negative qty is $4740.00.

Thus, the revenue increased by $145.50.

**Code Explanation:** First, convert the `qty` values to a number using `pd.to_numeric()`. Setting `errors='coerce'` means that any value that cannot be convered to a number is replaced with NaN. Next, count the number of missing (NaN) quantities and negative quantities.

For the missing-quantity step, first calculate the total revenue before dropping rows with missing qty. Then, keep only rows where qty is not missing and use `.copy()` to store the filtered result in the `clean` DataFrame. After that, caluclate the total revenue after dropping the missing qty and compute the change in revenue, storing it in `change_missing`. Finally, print the revenue before and after dropping mssing qty, along with the change in revenue, rounded to two decimal places.

For the negative-quantity step, first use the revenue after dropping missing qty (`after_missing`) as the revenue before dropping negative qty. Then, keep only rows where qty is greater than or equal to 0 and use `.copy()` to store the filtered result in the `clean` DataFrame. Next, caluclate the total revenue after dropping negative qty and compute the change in revenue, storing it in `change_negative`. Then, print the revenue before and after dropping negative qty, along with the change in revenue, rounded to two decimal places.

Finally, print the number of remaining rows and create two separate log entries to record the missing-quantity and negative-quantity cleaning steps and the number of rows affected.

### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [8]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(clean['item'].value_counts())
print('before:', sorted(str(x) for x in clean['item'].unique()))

before_count = clean['item'].nunique()
before_item = clean['item'].copy()

# ITEM_MAP = {...}
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}
clean['item'] = clean['item'].replace(ITEM_MAP)

# Count distinct item values after cleaning
after_count = clean['item'].nunique()
print('after: ', sorted(clean['item'].unique()))

print('Remaining', len(clean), 'rows')

rows_affected = (before_item != clean['item']).sum()

# create log(...)
log('canonicalize item', f'normalized item names from {before_count} distinct items to {after_count}', rows_affected)


item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
before: ['Cheeseburger', 'Foam Finger', 'Rain Poncho', 'cheese burger', 'foam finger', 'rain poncho']
after:  ['Cheeseburger', 'Foam Finger', 'Rain Poncho']
Remaining 275 rows
[canonicalize item] normalized item names from 6 distinct items to 3 (126 row(s))


**Code Explanation:** First, print the number of distinct values in the `item` column and list all item names before cleaning. Second, create an `ITEM_MAP` dictionary to standardize inconsistent item names by mapping '`Cheeseburger`' and '`cheese burger`' to '`Cheeseburger`', '`Foam Finger`' and '`foam finger`' to '`Foam Finger`', and '`Rain Poncho`' and '`rain poncho`' to '`Rain Poncho`'. Third, use `ITEM_MAP` to replace the values in the `item` column of the `clean` DataFrame. Fourth, count and print the number of distinct values in the `item` column after cleaning. Fifth, print the number of remaining rows, which should not change during this step. Then, compare the item values before and after cleaning, count how many rows changed, and store that number in `rows_affected`. Finally, create a log entry to record the cleaning step and the number of rows affected.

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [9]:
print(clean['category'].value_counts())
print('before:', sorted(str(x) for x in clean['category'].unique()))

before_count = clean['category'].nunique()
before_category = clean['category'].copy()

# CATEGORY_MAP = {...}
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)

after_count = clean['category'].nunique()

print('after: ', sorted(clean['category'].unique()))

print('Remaining', len(clean), 'rows')

rows_affected = (before_category != clean['category']).sum()

# TODO: log(...)
log('normalize category', f'normalized category names from {before_count} distinct categories to {after_count}; change Apparel to Merch', rows_affected)


category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Food', 'Merch', 'RainGear']
Remaining 275 rows
[normalize category] normalized category names from 6 distinct categories to 3; change Apparel to Merch (132 row(s))


**Code Explanation:** First, print the number of distinct values in the `category` column and list all category names before cleaning. Second, create a `CATEGORY_MAP` dictionary to standardize inconsistent category names by mapping '`Food`' and '`food`' to '`Food`', '`Merch`' and '`Apparel`' to '`Merch`', and '`RainGear`' and '`rain-gear`' to '`RainGear`'. I mapped `Apparel` to `Merch` because apparel can reasonably be treated as merchandise, so I combined them into the same `Merch` category. Third, use `CATEGORY_MAP` to replace the values in the `category` column of the `clean` DataFrame. Fourth, count and print the number of distinct values in the `category` column after cleaning. Fifth, print the number of remaining rows, which should not change during this step. Then, compare the category values before and after cleaning, count how many rows changed, and store that number in `rows_affected`. Finally, create a log entry to record the cleaning step and the number of rows affected.

### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [10]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
assert set(clean['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(clean['category'].unique()) == {'Food', 'Merch', 'RainGear'}
print('clean:', clean.shape)
clean

clean: (275, 5)


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,Foam Finger,Merch,1.0,7.5
2,2,Cheeseburger,Merch,1.0,7.5
4,4,Cheeseburger,Merch,1.0,7.5
5,5,Foam Finger,Food,3.0,6.0
...,...,...,...,...,...
294,294,Rain Poncho,Merch,2.0,6.0
295,295,Rain Poncho,Food,3.0,6.0
297,297,Foam Finger,Food,1.0,7.5
298,298,Rain Poncho,Merch,1.0,7.5


**Code Explanation:** First, create five `assert` statements to verify that the `clean` DataFrame has no duplicate rows, the minimum quantity is greater than or equal to 1, the `price` column has a `float` data type, the only distinct item names are `Cheeseburger`, `Foam Finger`, and `Rain Poncho`, and the only distinct category names are `Food`, `Merch`, and `RainGear`. Then, print the number of rows and columns using `.shape()` and display the `clean` DataFrame.

### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [11]:
# add a revenue column
clean['revenue'] = clean['price'] * clean['qty']
by_category = clean.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print(f'Total revenue: ${by_category.sum():.2f}')
by_category

Total revenue: $4740.00


,revenue
category,
Food,1656.0
Merch,1572.0
RainGear,1512.0


**What I would tell the vendor:** I would tell the vendor to stock more food because it generated the highest total revenue, $1656.00.

The total revenue is $4740.00.

**Code Explanation:** First, add a `revenue` column to the `clean` DataFrame by multiplying `price` by `qty`. Second, group the `clean` DataFrame by `category`, caluclate the total revenue for each category, sort the results in descending order from highest to lowest, and round the revenue values to two decimal places, storing the result in `by_category`. Finally, print the overall total revenue of `$4740.00` and display the `by_category` table.

### TODO 8 — read back your log

In [12]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop duplicates,dropped exact duplicate rows,15
1,clean price,price arrived as text; converted to float,300
2,drop missing qty,dropped rows with no quantity,12
3,drop negative qty,dropped rows with negative quantity,13
4,canonicalize item,normalized item names from 6 distinct items to 3,126
5,normalize category,normalized category names from 6 distinct cate...,132


**Code Explanation:** Convert the `DECISIONS` log into a pandas DataFrame to display all recorded cleaning steps, decisions, and the number of rows affected.

### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping duplicates changed my revenue total the most: `$4852.50` before -> `$4594.50` after, a decrease of `$258.00`.

b) Instead of dropping rows with negative `qty`, a reasonable person would keep rows with negative `qty` and interpret the the negative quantities as returns or refunds when calculating net revenue. If those rows were kept, the reported revenue would be `$4594.50` instead of `$4740.00`, which is `$145.50` lower.

I chose to drop the rows with negative `qty` because negative quantities do not make sense when treating `qty` as units sold, and removing them makes the dataset more appropriate for reporting gross sales revenue.